# Ingeniero de boxes para simracing

**Carlos Peñalver · Proyecto del módulo de IA Generativa**

Gemini, Chroma y LangGraph. Corpus indexado; agente y memoria pendientes.


## 0. Configuración

Usar el kernel de `.venv` de Proyecto. Configurar las credenciales en `Proyecto/.env`.


In [1]:
from pathlib import Path
import json
import os
import sys

from dotenv import load_dotenv
from langchain_core.documents import Document

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from retrieval import GeminiRetrievalEmbeddings, index_documents, search

load_dotenv(ROOT / ".env", override=True)
CORPUS_DIR = ROOT / "data/corpus"
CHROMA_DIR = ROOT / "data/chroma"
SYSTEM_PROMPT = (ROOT / "prompts/system.md").read_text(encoding="utf-8")


## 1. Corpus

75 fichas: LMU (Ultimate Setup Hub), iRacing (Chris Haye) y ACC (Driver61).
Cada ficha conserva el caso completo y sus metadatos. Detalle en `docs/FUENTES.md`.


## 2. Embeddings y Chroma

Mismo flujo que en clase: documentos → Gemini Embedding 2 → Chroma → búsqueda.


In [2]:
records = [json.loads(line) for line in
           (CORPUS_DIR / "mvp/preparado/corpus.jsonl").read_text(encoding="utf-8").splitlines()]
documents = [
    Document(page_content=record["text"], metadata={
        key: value for key, value in record.items() if key != "text"
    })
    for record in records
]


### 2.1. Embeddings de Gemini

`GeminiRetrievalEmbeddings` adapta el cliente de clase a Embedding 2. Detalle en `retrieval.py`.


In [3]:
MODEL_EMBEDDING = os.environ["GEMINI_EMBEDDING_MODEL"]
llm_embedding = GeminiRetrievalEmbeddings(
    model=MODEL_EMBEDDING,
    api_key=os.environ["GOOGLE_API_KEY"],
    vertexai=False,
)


### 2.2. Índice persistente

Se guarda en `data/chroma/`. El mismo corpus reutiliza los embeddings existentes.


In [4]:
vectorstore = index_documents(documents, llm_embedding, CHROMA_DIR)


### 2.3. Búsqueda

Recuperar tres fichas del simulador elegido. Esta celda consume un embedding de consulta.


In [ ]:
simulator = "lmu"
query = "En LMU, tengo sobreviraje al acelerar a la salida de una curva lenta. ¿Qué propone la guía?"
results = search(vectorstore, query, simulator, k=3)

for doc in results:
    print(doc.metadata["id"], "—", doc.metadata["title"])


## 3. System prompt

Disponible en `prompts/system.md` y cargado en `SYSTEM_PROMPT`. Justificación en el README.


## 4. Agente en LangGraph — pendiente

Pregunta → recuperación por simulador → respuesta de Gemini con fuentes.


## 5. Memoria — pendiente

Historial por conversación: síntomas, cambios confirmados y observaciones del piloto.


## 6. Chat y cinco ejemplos documentados — pendiente

Crear una celda de chat interactivo con una salida clara, por ejemplo `/salir`. Después, registrar al menos cinco preguntas y sus respuestas **reales**, con fuentes y un seguimiento. No sustituir las respuestas por expectativas.

| Caso | Pregunta prevista | Respuesta real y fuentes |
| --- | --- | --- |
| E01 | Diagnóstico inicial sin contexto suficiente | Pendiente de ejecución |
| E02 | iRacing: síntoma y prueba de ajuste documentada | Pendiente de ejecución |
| E03 | ACC: consulta documental sobre pianos | Pendiente de ejecución |
| E04 | LMU: sobreviraje por fase y velocidad | Pendiente de ejecución |
| E05 | Seguimiento de E02: cambio aplicado y nuevo síntoma | Pendiente de ejecución |

Usar el corpus elegido y los casos de docs/EVALUACION.md. Identificar los escenarios ficticios del piloto y añadir casos sin evidencia, setup fijo y cambio de simulador. Las respuestas de Gemini se documentarán tras ejecutarlas; no equivalen a validación en pista.


## 7. Comparación con Gemini sin RAG — pendiente

Mismas preguntas, modelo e historial; comparar las respuestas reales con los criterios de `docs/EVALUACION.md`.


## 8. Bonus: Streamlit y presentación — pendiente

Chat web sobre el mismo núcleo y demo de 5–10 minutos.
